In [1]:
import torch
from torch import nn
from torch import optim
import numpy as np
import pickle
from matplotlib import pyplot as plt
from matplotlib_inline import backend_inline


def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)


def cpu():
    """Get the CPU device.

    Defined in :numref:`sec_use_gpu`"""
    return torch.device('cpu')


def gpu(i=0):
    """Get a GPU device.

    Defined in :numref:`sec_use_gpu`"""
    return torch.device(f'cuda:{i}')


def num_gpus():
    """Get the number of available GPUs.

    Defined in :numref:`sec_use_gpu`"""
    return torch.cuda.device_count()


def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu().

    Defined in :numref:`sec_use_gpu`"""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()


class NNFrameWork(nn.Module):
    """
    通用神经网络训练框架
    核心框架结构梳理：

    >训练类方法:
    train_real_time(x, target) -> loss
    train_fixed(data_iter, epochs, [save_path, plot_hist, other_options]) -> loss <from last time>
        >训练函数接口(interface):
        _train_fixed_implementation
        >训练核心组件:
        _check_feasibility
        _get_gradient
        _self_update

    >模型衡量/应用方法:
    predict(with gradient/without gradient)
    evaluate_loss
    evaluate_accuracy

    >模型信息输入输出:
    plot_hist
    save_model
    load_model
    load_optimizer

    >模型基本架构设置:
    set_optimizer
    set_weights_init
    set_criterion
    set_device
    """

    def __init__(self):
        super().__init__()
        ################################
        self.optimizer = None
        self.criterion = None
        self.device = None
        self.clip_value = None
        self.init_fs = None
        #################################
        self.iter_times = 0  # input_num
        self.loss_history = []
        self.temp_loss_history = []
        self.to(self.device)

    def train_real_time(self, x, target):
        self._check_feasibility()
        self.train()

        # 更新队列及迭代次数
        self.iter_times += 1

        # 学习及更新
        loss = self._get_gradient(x, target)
        self._self_update()

        # 学习历史记录添加
        self.loss_history.append(loss)

    def train_fixed(self, data_iter, epochs, save_path=None, plot_hist=True,
                    other_options=None):
        self._check_feasibility()
        self.train()
        loss = self._train_fixed_implementation(data_iter, epochs, other_options)
        ###############################save and plot hist##############################
        if save_path is not None:
            torch.save(self.state_dict(), save_path + ".pt")
            torch.save(self.optimizer.state_dict(), save_path + '_optimizer_state.pth')
            with open(save_path + '_train_hist.pkl', 'wb') as f:
                pickle.dump(self.loss_history, f)
        if plot_hist:
            self.plot_hist()
        ###############################save and plot hist##############################
        return loss

    def _train_fixed_implementation(self, data_iter, epochs, other_options=None):

        loss_threshold = None  # loss阈值，用于判断迭代退出条件
        if other_options is not None:
            loss_threshold = other_options['loss_threshold']

        for epoch in range(epochs):
            for i, (x, y) in enumerate(data_iter):
                x = x.to(self.device)
                y = y.to(self.device)
                ##################training################
                # 学习及更新
                loss = self._get_gradient(x, y)
                if loss_threshold is not None:  # 判断loss是否已经达到要求
                    if loss < loss_threshold:
                        return loss
                self._self_update(x, y)
                # 学习历史记录添加
                if i == len(data_iter) - 1:
                    self.temp_loss_history.append(loss)
                    self.loss_history.append(np.mean(self.temp_loss_history))
                    self.temp_loss_history = []
                    # 更新队列及迭代次数
                    self.iter_times += 1
                else:
                    self.temp_loss_history.append(loss)
                ##################training################
        return self.loss_history[-1]

    def _check_feasibility(self):
        assert self.optimizer is not None, "No optimizer specified!"
        assert self.criterion is not None, "No criterion specified!"
        assert self.device is not None, "No device specified!"

    def _get_gradient(self, x, target):
        x = x.to(self.device)
        target = target.to(self.device)

        # 学习及更新
        output = self(x)
        loss = self.criterion(output, target)
        self.optimizer.zero_grad()
        loss.backward()
        return loss.item()

    def _self_update(self, data=None, target=None):
        if self.clip_value is not None:
            nn.utils.clip_grad_norm_(self.parameters(), self.clip_value)  # 梯度截断
        self.optimizer.step()

    def predict(self, x, no_grad=False):
        self.eval()
        if no_grad:
            with torch.no_grad():
                return self(x)
        return self(x)

    def evaluate_loss(self, data_iter):
        loss_list = []
        for i, (x, y) in enumerate(data_iter):
            x = x.to(self.device)
            y = y.to(self.device)
            ##################evaluate################
            with torch.no_grad():
                # 计算loss
                output = self.predict(x)
                loss = self.criterion(output, y)
                loss_list.append(loss.cpu())
            ##################evaluate################
        return np.mean(loss_list)

    def evaluate_accuracy(self, data_iter):
        self.eval()  # 将模型设置为评估模式
        correct = 0
        total = 0
        with torch.no_grad():  # 在评估模式下不需要计算梯度
            for data in data_iter:
                inputs, labels = data[0].to(self.device), data[1].to(self.device)  # 移动数据到设备（如GPU）
                outputs = self.predict(inputs, no_grad=True)
                _, predicted = torch.max(outputs, 1)  # 获取每行最大值的位置（即预测的类别）
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        return accuracy

    def plot_hist(self):
        plt.figure()
        backend_inline.set_matplotlib_formats('svg')
        plt.plot(self.loss_history)
        plt.xlabel("epochs")
        plt.ylabel("loss")
        plt.show()

    def save_model(self, model_name="model"):
        torch.save(self.state_dict(), model_name + ".pt")
        torch.save(self.optimizer.state_dict(), model_name + '_optimizer_state.pth')
        with open(model_name + '_train_hist.pkl', 'wb') as f:
            pickle.dump(self.loss_history, f)

    def load_model(self, model_path_without_suffix):
        self.load_state_dict(torch.load(model_path_without_suffix + ".pt"))
        with open(model_path_without_suffix + '_train_hist.pkl', 'rb') as f:
            self.loss_history = pickle.load(f)

    def load_optimizer(self, model_path_without_suffix):
        self.optimizer.load_state_dict(torch.load(model_path_without_suffix + '_optimizer_state.pth'))

    def set_optimizer(self, lr=0.01, momentum=0.0, clip_value=float("inf"), optimizer="default"):
        self.clip_value = clip_value
        if optimizer == "default":
            self.optimizer = optim.SGD(self.get_param_groups(), lr=lr, momentum=momentum)
        else:
            self.optimizer = optimizer
        print(self.optimizer)

    def get_param_groups(self):
        # 逻辑存在问题，因为按照以下逻辑，大块层永远会覆盖小块层的学习率，当大块层没有学习率时，小块层也将没有学习率
        # param_groups = [
        #     {
        #         'params': module.parameters(),
        #         'lr': module.lr
        #     } if hasattr(module, 'lr') else
        #     {
        #         'params': module.parameters()
        #     }
        #     for module in self.modules() if isinstance(module, NNFrameWork)
        # ]
        # return param_groups
        added_params = set()  # 用于追踪已经添加的参数
        processed_modules = set()  # 用于追踪已经处理的模块
        param_groups = []

        for module in self.modules():
            if module in processed_modules:
                continue  # 如果模块已经处理，跳过

            if isinstance(module, NNFrameWork):
                params = set(module.parameters())  # 将参数转换为集合以确保唯一性
                if params & added_params:
                    continue  # 如果已经添加过，跳过

                added_params.update(params)  # 添加新参数

                if hasattr(module, 'lr') and module.lr is not None:
                    param_group = {
                        'params': params,
                        'lr': module.lr
                    }
                else:
                    param_group = {
                        'params': params
                    }

                param_groups.append(param_group)

                # 标记该模块及其子模块为已处理
                processed_modules.update(module.get_submodules())  # 假设get_submodules()返回子模块

        return param_groups

    def set_weights_init(self, init_f):
        self.init_fs.append(init_f)
        self.apply(init_f)

    def set_criterion(self, criterion):
        self.criterion = criterion

    def set_device(self, device=cpu()):
        self.device = device
        self.to(self.device)

In [2]:
optim.Adam

torch.optim.adam.Adam

In [3]:
class A:
    def __init__(self):
        self.a = 1

In [4]:
class B(A):
    def __init__(self):
        super().__init__()
        self.a = 2

In [5]:
k = B()
k.a

2

In [6]:
l1 = nn.Linear(2, 2)
l2 = nn.Linear(2, 2)
net = nn.Sequential(l1, l2)
net = nn.Sequential(net, l1, l2)
for idx, m in enumerate(net.modules()):
    print(idx, '->', m)
#
# 0 -> Sequential(
#               (0): Linear(in_features=2, out_features=2, bias=True)
#               (1): Linear(in_features=2, out_features=2, bias=True)
#             )
#             1 -> Linear(in_features=2, out_features=2, bias=True)

0 -> Sequential(
  (0): Sequential(
    (0): Linear(in_features=2, out_features=2, bias=True)
    (1): Linear(in_features=2, out_features=2, bias=True)
  )
  (1): Linear(in_features=2, out_features=2, bias=True)
  (2): Linear(in_features=2, out_features=2, bias=True)
)
1 -> Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Linear(in_features=2, out_features=2, bias=True)
)
2 -> Linear(in_features=2, out_features=2, bias=True)
3 -> Linear(in_features=2, out_features=2, bias=True)


In [7]:
for child in net.children():
    print(child)

Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Linear(in_features=2, out_features=2, bias=True)
)
Linear(in_features=2, out_features=2, bias=True)
Linear(in_features=2, out_features=2, bias=True)


In [10]:
for name, m in net.named_parameters():
    print(name)
    print(m)

0.0.weight
Parameter containing:
tensor([[-0.1051, -0.0873],
        [ 0.5107, -0.3170]], requires_grad=True)
0.0.bias
Parameter containing:
tensor([-0.3745, -0.1616], requires_grad=True)
0.1.weight
Parameter containing:
tensor([[ 0.6310,  0.0675],
        [-0.1020,  0.4926]], requires_grad=True)
0.1.bias
Parameter containing:
tensor([-0.5253, -0.2776], requires_grad=True)


In [13]:
for module in net.modules():
    print(module)

Sequential(
  (0): Sequential(
    (0): Linear(in_features=2, out_features=2, bias=True)
    (1): Linear(in_features=2, out_features=2, bias=True)
  )
  (1): Linear(in_features=2, out_features=2, bias=True)
  (2): Linear(in_features=2, out_features=2, bias=True)
)
Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Linear(in_features=2, out_features=2, bias=True)
)
Linear(in_features=2, out_features=2, bias=True)
Linear(in_features=2, out_features=2, bias=True)


In [10]:
optim.SGD([  # {'params': net[0].parameters(), 'lr': 1e-2},
    {'params': net[1].parameters(), 'lr': 1e-2},
    {'params': net[2].parameters()}
], lr=1e-3, momentum=0.9)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0

Parameter Group 1
    dampening: 0
    differentiable: False
    foreach: None
    lr: 0.001
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)

In [208]:
# 核心思想：迭代从网络架构中找出网络组件，当网络中存在与父节点不一致的学习率时，将架构分开拆分
# 函数的接口定义：
# 输入：模型 -> 输出：params_list(包括了学习率), 是否存在不同的状态变量
# 递归的三要素：递归元素之间的相互逻辑、递归传递状态的维护、递归的终点
def get_params(model, lr_predecessor=None):
    params_list = []  # 参数与学习率列表
    status = False  # 指示在该模型的子模块内是否存在其它学习率的设置
    lr = lr_predecessor  # 前继节点的设置学习率

    if hasattr(model, 'lr') and model.lr is not None:  # 判断该节点是否存在学习率设置
        status = True
        lr = model.lr

    for child in model.children():
        params_list_child, status_child = get_params(child, lr)
        if not params_list_child:  # 判断child是否为最底层的节点(当child没有子节点时，其不进入循环，返回列表为空)
            if status_child:  # 判断底层节点是否有自定义的学习率
                params_list_child = [{'params': child.parameters(), 'lr': child.lr}]
            elif lr is not None:  # 判断父节点是否有传递的学习率
                params_list_child = [{'params': child.parameters(), 'lr': lr}]
            else:  # 无学习率要求
                params_list_child = [{'params': child.parameters()}]
        if status_child:  # 只要子网络组分中存在一个模型有学习率设置，则需要拆分网络模块表以添加该学习率
            status = True
        params_list.extend(params_list_child)

    if not status and lr is None:  # 表明该模型中没有找到不同的学习率组件且没有指定的前继学习率
        return [{'params': model.parameters()}], status

    return params_list, status

In [209]:
class A(nn.Module):
    def __init__(self):
        super().__init__()
        self.lr = 0.1


class B(nn.Module):
    def __init__(self):
        super().__init__()


class C(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = A()
        self.b = B()
        self.lr = 0.5


seq = nn.Sequential(A(), C(), B())

In [210]:
get_params(C())

([{'params': <generator object Module.parameters at 0x0000019173130EB0>,
   'lr': 0.1},
  {'params': <generator object Module.parameters at 0x0000019173131EE0>,
   'lr': 0.5}],
 True)

In [179]:
get_params(B())

([{'params': <generator object Module.parameters at 0x0000019172ACB5A0>}],
 False)

In [180]:
B().children()
print(list(B().children()))
print(list(A().children()))

[]
[]


In [181]:
k = A()
for i in k.children():
    print(i)

In [182]:
p_list, status = get_params(seq)
print(p_list)
print(status)

[{'params': <generator object Module.parameters at 0x0000019172ACBE60>, 'lr': 0.1}, {'params': <generator object Module.parameters at 0x0000019172ACC040>, 'lr': 0.1}, {'params': <generator object Module.parameters at 0x0000019172ACC0B0>, 'lr': 0.5}, {'params': <generator object Module.parameters at 0x0000019172ACBED0>}]
True


In [67]:
a11 = [1, 2, 3]
b11 = [4, 5, 6]
a11.extend(b11)

In [68]:
a11

[1, 2, 3, 4, 5, 6]

In [121]:
get_params(net)

None
None
None
None
None
None


([{'params': <generator object Module.parameters at 0x0000019172A6CCF0>}],
 False)

In [127]:
print(C().lr)

0.5


In [204]:
import torch
import torch.nn as nn

class D(nn.Module):
    def __init__(self):
        super(D, self).__init__()
        self.weights = torch.randn(3, requires_grad=True)

    def forward(self, x):
        # 假设 x 和 self.weights 都是一维的，并且长度相同
        return x * self.weights

# 实例化模型，并设置为评估模式
d1 = D()
d2 = D()
d2.eval()  # 设置 d2 模块为评估模式

# 使用 nn.Sequential 创建一个包含 d1 和 d2 的序列模型
d3 = nn.Sequential(d1, d2)

# 创建一个输入张量，并设置 requires_grad=True
x = torch.tensor([1.1, 2.1, 3.1], requires_grad=True)

# 计算损失
l = d3(x)
print(l)

# 反向传播计算梯度
l.sum().backward()  # 使用 l.sum() 来确保我们有一个标量值，可以求导
d1.weights.grad

tensor([ 0.3823, -2.4619, -0.3709], grad_fn=<MulBackward0>)


tensor([ 1.0477,  2.2307, -0.9933])

In [205]:
d2.weights.grad

tensor([ 0.4014, -2.3176,  1.1575])

In [206]:
import torch
from torch import nn
from torch import optim
import numpy as np
import pickle
from matplotlib import pyplot as plt
from matplotlib_inline import backend_inline


def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)


def cpu():
    """Get the CPU device.

    Defined in :numref:`sec_use_gpu`"""
    return torch.device('cpu')


def gpu(i=0):
    """Get a GPU device.

    Defined in :numref:`sec_use_gpu`"""
    return torch.device(f'cuda:{i}')


def num_gpus():
    """Get the number of available GPUs.

    Defined in :numref:`sec_use_gpu`"""
    return torch.cuda.device_count()


def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu().

    Defined in :numref:`sec_use_gpu`"""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()


class NNFrameWork(nn.Module):
    """
    通用神经网络训练框架
    核心框架结构梳理：

    >训练类方法:
    train_real_time(x, target) -> loss
    train_fixed(data_iter, epochs, [save_path, plot_hist, other_options]) -> loss <from last time>
        >训练函数接口(interface):
        _train_fixed_implementation
        >训练核心组件:
        _check_feasibility
        _get_gradient
        _self_update

    >模型衡量/应用方法:
    predict(with gradient/without gradient)
    evaluate_loss
    evaluate_accuracy

    >模型信息输入输出:
    plot_hist
    save_model
    load_model
    load_optimizer

    >模型基本架构设置:
    set_optimizer
    set_weights_init
    set_criterion
    set_device
    """

    def __init__(self):
        super().__init__()
        ################################
        self.optimizer = None
        self.criterion = None
        self.device = None
        self.clip_value = None
        self.init_fs = None
        #################################
        self.iter_times = 0  # input_num
        self.loss_history = []
        self.temp_loss_history = []
        self.to(self.device)

    def train_real_time(self, x, target):
        self._check_feasibility()
        self.train()

        # 更新队列及迭代次数
        self.iter_times += 1

        # 学习及更新
        loss = self._get_gradient(x, target)
        self._self_update()

        # 学习历史记录添加
        self.loss_history.append(loss)

    def train_fixed(self, data_iter, epochs, save_path=None, plot_hist=True,
                    other_options=None):
        self._check_feasibility()
        self.train()
        loss = self._train_fixed_implementation(data_iter, epochs, other_options)
        ###############################save and plot hist##############################
        if save_path is not None:
            torch.save(self.state_dict(), save_path + ".pt")
            torch.save(self.optimizer.state_dict(), save_path + '_optimizer_state.pth')
            with open(save_path + '_train_hist.pkl', 'wb') as f:
                pickle.dump(self.loss_history, f)
        if plot_hist:
            self.plot_hist()
        ###############################save and plot hist##############################
        return loss

    def _train_fixed_implementation(self, data_iter, epochs, other_options=None):

        loss_threshold = None  # loss阈值，用于判断迭代退出条件
        if other_options is not None:
            loss_threshold = other_options['loss_threshold']

        for epoch in range(epochs):
            for i, (x, y) in enumerate(data_iter):
                x = x.to(self.device)
                y = y.to(self.device)
                ##################training################
                # 学习及更新
                loss = self._get_gradient(x, y)
                if loss_threshold is not None:  # 判断loss是否已经达到要求
                    if loss < loss_threshold:
                        return loss
                self._self_update(x, y)
                # 学习历史记录添加
                if i == len(data_iter) - 1:
                    self.temp_loss_history.append(loss)
                    self.loss_history.append(np.mean(self.temp_loss_history))
                    self.temp_loss_history = []
                    # 更新队列及迭代次数
                    self.iter_times += 1
                else:
                    self.temp_loss_history.append(loss)
                ##################training################
        return self.loss_history[-1]

    def _check_feasibility(self):
        assert self.optimizer is not None, "No optimizer specified!"
        assert self.criterion is not None, "No criterion specified!"
        assert self.device is not None, "No device specified!"

    def _get_gradient(self, x, target):
        x = x.to(self.device)
        target = target.to(self.device)

        # 学习及更新
        output = self(x)
        loss = self.criterion(output, target)
        self.optimizer.zero_grad()
        loss.backward()
        return loss.item()

    def _self_update(self, data=None, target=None):
        if self.clip_value is not None:
            nn.utils.clip_grad_norm_(self.parameters(), self.clip_value)  # 梯度截断
        self.optimizer.step()

    def predict(self, x, no_grad=False):
        self.eval()
        if no_grad:
            with torch.no_grad():
                return self(x)
        return self(x)

    def evaluate_loss(self, data_iter):
        loss_list = []
        for i, (x, y) in enumerate(data_iter):
            x = x.to(self.device)
            y = y.to(self.device)
            ##################evaluate################
            with torch.no_grad():
                # 计算loss
                output = self.predict(x)
                loss = self.criterion(output, y)
                loss_list.append(loss.cpu())
            ##################evaluate################
        return np.mean(loss_list)

    def evaluate_accuracy(self, data_iter):
        self.eval()  # 将模型设置为评估模式
        correct = 0
        total = 0
        with torch.no_grad():  # 在评估模式下不需要计算梯度
            for data in data_iter:
                inputs, labels = data[0].to(self.device), data[1].to(self.device)  # 移动数据到设备（如GPU）
                outputs = self.predict(inputs, no_grad=True)
                _, predicted = torch.max(outputs, 1)  # 获取每行最大值的位置（即预测的类别）
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        return accuracy

    def plot_hist(self):
        plt.figure()
        backend_inline.set_matplotlib_formats('svg')
        plt.plot(self.loss_history)
        plt.xlabel("epochs")
        plt.ylabel("loss")
        plt.show()

    def save_model(self, model_name="model"):
        torch.save(self.state_dict(), model_name + ".pt")
        torch.save(self.optimizer.state_dict(), model_name + '_optimizer_state.pth')
        with open(model_name + '_train_hist.pkl', 'wb') as f:
            pickle.dump(self.loss_history, f)

    def load_model(self, model_path_without_suffix):
        self.load_state_dict(torch.load(model_path_without_suffix + ".pt"))
        with open(model_path_without_suffix + '_train_hist.pkl', 'rb') as f:
            self.loss_history = pickle.load(f)

    def load_optimizer(self, model_path_without_suffix):
        self.optimizer.load_state_dict(torch.load(model_path_without_suffix + '_optimizer_state.pth'))

    def set_optimizer(self, lr=0.01, momentum=0.0, clip_value=float("inf"), optimizer="default"):
        self.clip_value = clip_value
        if optimizer == "default":
            self.optimizer = optim.SGD(self.get_param_groups(), lr=lr, momentum=momentum)
        else:
            self.optimizer = optimizer
        print(self.optimizer)

    def get_param_groups(self):
        return get_params(self)

    def set_weights_init(self, init_f):
        self.init_fs.append(init_f)
        self.apply(init_f)

    def set_criterion(self, criterion):
        self.criterion = criterion

    def set_device(self, device=cpu()):
        self.device = device
        self.to(self.device)

In [217]:
class Sequential(nn.Sequential, NNFrameWork):
    pass

In [218]:
class A(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.lr = 0.5
        self.b = B()
        self.c = C()
        self.d = D()

class B(nn.Module):
    def __init__(self):
        super().__init__()
        self.lr = 0.1
        self.c = C()
        self.d = D()

class C(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.lr = 0.8

class D(nn.Module):
    def __init__(self):
        super().__init__()

q = Sequential(D(),A(),B(),C())
get_params(q)

([{'params': <generator object Module.parameters at 0x0000019173133B50>},
  {'params': <generator object Module.parameters at 0x0000019173133F40>,
   'lr': 0.8},
  {'params': <generator object Module.parameters at 0x0000019173133DF0>,
   'lr': 0.1},
  {'params': <generator object Module.parameters at 0x0000019173133E60>,
   'lr': 0.8},
  {'params': <generator object Module.parameters at 0x0000019173133ED0>,
   'lr': 0.5},
  {'params': <generator object Module.parameters at 0x00000191731E4040>,
   'lr': 0.8},
  {'params': <generator object Module.parameters at 0x00000191731E40B0>,
   'lr': 0.1},
  {'params': <generator object Module.parameters at 0x0000019173133CA0>,
   'lr': 0.8}],
 True)

In [219]:
q.get_param_groups()

([{'params': <generator object Module.parameters at 0x00000191731E4190>},
  {'params': <generator object Module.parameters at 0x00000191731E4510>,
   'lr': 0.8},
  {'params': <generator object Module.parameters at 0x00000191731E4580>,
   'lr': 0.1},
  {'params': <generator object Module.parameters at 0x00000191731E4430>,
   'lr': 0.8},
  {'params': <generator object Module.parameters at 0x00000191731E44A0>,
   'lr': 0.5},
  {'params': <generator object Module.parameters at 0x00000191731E45F0>,
   'lr': 0.8},
  {'params': <generator object Module.parameters at 0x00000191731E4660>,
   'lr': 0.1},
  {'params': <generator object Module.parameters at 0x00000191731E4350>,
   'lr': 0.8}],
 True)